# Road Sign Detection - YOLO Fine-Tuning (Google Colab)

Companion to `rag_pipeline.ipynb`. This notebook fine-tunes **`yolo11n`** on a road-sign dataset and
produces `best.pt`, which the FastAPI backend loads for image queries.

**Why Colab?** Fine-tuning is the only genuinely GPU-bound step in this project. A free **T4 (16 GB)**
trains faster than a 6 GB laptop GPU and does not tie up your machine. Everything else - the vector
store, the API, the frontend and the Ollama LLM - runs locally.

### Before you start
1. **Runtime -> Change runtime type -> T4 GPU.** Confirm with the first cell.
2. You need a free **Roboflow** account for the dataset export (it supplies YOLO-format labels; the
   original Kaggle release is PASCAL VOC XML, which YOLO cannot read).

### What you do with the output
Download `best.pt` at the end and save it to `backend/models/signs_yolo.pt` in the repo.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%pip install -q ultralytics roboflow
import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| CUDA:", torch.cuda.is_available())

## 1. Dataset

**Kaggle Road Sign Detection** (`andrewmvd/road-sign-detection`) - 877 images, 4 classes:
`stop`, `speedlimit`, `crosswalk`, `trafficlight`. Every one of those is documented in all three
driver handbooks, so each class maps cleanly onto a handbook lookup.

Open the Roboflow Universe mirror, choose **Download this Dataset -> YOLOv8**, and paste the snippet
Roboflow gives you below. Enable augmentation (flip, brightness, blur) at export: ~219 images per
class is workable but benefits from 2x augmentation.

In [ ]:
from roboflow import Roboflow

# Replace with the snippet from Roboflow ("Download this Dataset" -> YOLOv8 -> "show download code").
# It looks like:
#   rf = Roboflow(api_key="YOUR_KEY")
#   dataset = rf.workspace("kaggle-road-sign-dataset").project("kaggle-road-sign-dataset").version(1).download("yolov8")
rf = Roboflow(api_key="YOUR_API_KEY_HERE")
project = rf.workspace("kaggle-road-sign-dataset").project("kaggle-road-sign-dataset")
dataset = project.version(1).download("yolov8")

DATA_YAML = f"{dataset.location}/data.yaml"
print("data.yaml ->", DATA_YAML)

In [ ]:
# Always check what you actually downloaded before spending GPU time on it.
import yaml, collections, pathlib

cfg = yaml.safe_load(open(DATA_YAML))
print("classes:", cfg["names"])

for split in ("train", "valid", "test"):
    d = pathlib.Path(dataset.location) / split / "labels"
    if not d.exists():
        continue
    counts = collections.Counter()
    for f in d.glob("*.txt"):
        for line in f.read_text().splitlines():
            if line.strip():
                counts[cfg["names"][int(line.split()[0])]] += 1
    n_img = len(list((d.parent / "images").glob("*")))
    print(f"  {split:6s} {n_img:5d} images  {dict(counts)}")

**Check the class balance above.** If a class has only a handful of instances, its mAP will be noise -
say so in the write-up rather than quietly reporting a single headline number.

## 2. Fine-tune

`yolo11n` is the smallest YOLO11 variant: ~6 MB of weights, which keeps the repo well under the
50 MB guidance and runs in ~60 ms per image on CPU at serve time.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")          # COCO-pretrained starting point
results = model.train(
    data=DATA_YAML,
    epochs=60,
    imgsz=640,
    batch=32,                       # fits comfortably on a 16 GB T4
    patience=15,                    # stop early if val mAP plateaus
    project="runs", name="signs",
    seed=0,                         # reproducible
)

In [ ]:
metrics = model.val()
print(f"mAP50    : {metrics.box.map50:.3f}")
print(f"mAP50-95 : {metrics.box.map:.3f}")
print("\nper class:")
for i, name in enumerate(model.names.values()):
    print(f"  {name:14s} mAP50={metrics.box.ap50[i]:.3f}")

Report these numbers honestly in the README. With ~880 images over 4 classes, mAP50 in the 0.7-0.9
range is a realistic result; tuning until one number looks impressive is not the point of the exercise.

In [ ]:
# Training curves and confusion matrix -- save these into docs/screenshots/ for the README.
from IPython.display import Image, display
for f in ("results.png", "confusion_matrix.png", "val_batch0_pred.jpg"):
    p = f"{results.save_dir}/{f}"
    try:
        display(Image(filename=p, width=820))
    except Exception:
        print("missing:", p)

## 3. Download the weights

Save the downloaded file as **`backend/models/signs_yolo.pt`** in the repo. The backend picks it up
automatically on its next start; `GET /health` will then report `"yolo_loaded": true`.

In [ ]:
from google.colab import files
import shutil

best = f"{results.save_dir}/weights/best.pt"
shutil.copy(best, "signs_yolo.pt")
print("size:", round(__import__("os").path.getsize("signs_yolo.pt") / 1e6, 1), "MB")

files.download("signs_yolo.pt")
# Also grab the curves for the README:
shutil.make_archive("training_plots", "zip", results.save_dir)
files.download("training_plots.zip")

> **Colab sessions are ephemeral.** Download `best.pt` as soon as training finishes - if the runtime
> disconnects you lose the weights and have to train again.